In [ ]:
# Enhanced Molecular Similarity Analysis Tool for Google Colab
# Step-by-step implementation with proper file handling and visualization saving

# =============================================================================
# STEP 1: Install Required Packages
# =============================================================================

!pip install rdkit-pypi matplotlib seaborn numpy pandas scikit-learn tqdm ipywidgets -q
print("✅ All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.5 MB/s eta 0:00:00
✅ All packages installed successfully!


In [ ]:
# =============================================================================
# STEP 2: Import Libraries and Setup
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Draw, Descriptors
from rdkit.DataStructs import TanimotoSimilarity, DiceSimilarity, ConvertToNumpyArray
from google.colab import files
import io
import os
import shutil
from tqdm.notebook import tqdm
from datetime import datetime
import warnings
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LinearSegmentedColormap

# Suppress warnings
warnings.filterwarnings('ignore')

# Create output directory
output_dir = 'similarity_analysis_results'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Output directory created: {output_dir}")

✅ Output directory created: similarity_analysis_results


In [ ]:
# =============================================================================
# STEP 3: Configure Matplotlib for High-Quality Output
# =============================================================================

# Set matplotlib backend and style for better file saving
plt.style.use('default')  # Use default instead of seaborn to avoid conflicts
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 18,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.2,
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.5,
    'lines.linewidth': 2.0,
    'axes.grid': True,
    'axes.axisbelow': True,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

# Define color schemes
COLORMAP_OPTIONS = {
    'viridis': plt.cm.viridis,
    'plasma': plt.cm.plasma,
    'coolwarm': plt.cm.coolwarm,
    'RdYlBu_r': plt.cm.RdYlBu_r,
    'custom_blue_red': LinearSegmentedColormap.from_list('custom',
                      ['#053061', '#4393c3', '#f7f7f7', '#f4a582', '#67001f'], N=100)
}

print("✅ Matplotlib configured for high-quality output")

✅ Matplotlib configured for high-quality output


In [ ]:
# =============================================================================
# STEP 4: Fingerprint Calculation Functions
# =============================================================================

def calculate_morgan_fingerprint(mol, radius=2, nBits=2048):
    """Calculate Morgan fingerprint (ECFP equivalent)."""
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)

def calculate_fcfp_fingerprint(mol, radius=2, nBits=2048):
    """Calculate FCFP fingerprint (Morgan with useFeatures=True)."""
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits, useFeatures=True)

def calculate_maccs_keys(mol):
    """Calculate MACCS keys fingerprint."""
    if mol is None:
        return None
    return MACCSkeys.GenMACCSKeys(mol)

def calculate_similarity(ref_mol, comp_mol):
    """Calculate comprehensive similarity metrics."""
    # Calculate fingerprints
    ref_morgan = calculate_morgan_fingerprint(ref_mol)
    comp_morgan = calculate_morgan_fingerprint(comp_mol)
    ref_fcfp = calculate_fcfp_fingerprint(ref_mol)
    comp_fcfp = calculate_fcfp_fingerprint(comp_mol)
    ref_maccs = calculate_maccs_keys(ref_mol)
    comp_maccs = calculate_maccs_keys(comp_mol)

    # Calculate similarities
    similarities = {
        'Tanimoto_Morgan': TanimotoSimilarity(ref_morgan, comp_morgan),
        'Tanimoto_FCFP': TanimotoSimilarity(ref_fcfp, comp_fcfp),
        'Tanimoto_MACCS': TanimotoSimilarity(ref_maccs, comp_maccs),
        'Dice_Morgan': DiceSimilarity(ref_morgan, comp_morgan),
        'Dice_FCFP': DiceSimilarity(ref_fcfp, comp_fcfp),
        'Dice_MACCS': DiceSimilarity(ref_maccs, comp_maccs)
    }

    # Calculate average similarity
    similarities['Average_Similarity'] = np.mean(list(similarities.values()))

    return similarities

print("✅ Fingerprint calculation functions defined")


✅ Fingerprint calculation functions defined


In [ ]:
# =============================================================================
# STEP 5: Visualization Functions with Proper File Saving
# =============================================================================

def save_figure(fig, filename, dpi=300):
    """Save figure with proper error handling."""
    try:
        filepath = os.path.join(output_dir, filename)
        fig.savefig(filepath, dpi=dpi, bbox_inches='tight',
                   facecolor='white', edgecolor='none', format='png')
        plt.close(fig)  # Close figure to free memory
        print(f"✅ Saved: {filename}")
        return True
    except Exception as e:
        print(f"❌ Error saving {filename}: {str(e)}")
        return False

def visualize_parent_molecule(smiles, title="Parent Molecule"):
    """Create and save parent molecule visualization."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    # Calculate molecular properties
    mol_wt = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    donors = Descriptors.NumHDonors(mol)
    acceptors = Descriptors.NumHAcceptors(mol)
    rotatable = Descriptors.NumRotatableBonds(mol)

    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # Draw molecule
    img = Draw.MolToImage(mol, size=(400, 400))
    ax1.imshow(img)
    ax1.axis('off')
    ax1.set_title(title, fontsize=16, fontweight='bold', pad=20)

    # Properties table
    ax2.axis('off')
    properties = [
        ['Property', 'Value'],
        ['Molecular Weight', f'{mol_wt:.2f}'],
        ['LogP', f'{logp:.2f}'],
        ['H-Bond Donors', f'{donors}'],
        ['H-Bond Acceptors', f'{acceptors}'],
        ['Rotatable Bonds', f'{rotatable}'],
        ['SMILES', smiles[:50] + '...' if len(smiles) > 50 else smiles]
    ]

    table = ax2.table(cellText=properties[1:], colLabels=properties[0],
                     cellLoc='left', loc='center', colWidths=[0.4, 0.6])
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    # Style the table
    for i in range(len(properties)):
        if i == 0:  # Header
            table[(i, 0)].set_facecolor('#4CAF50')
            table[(i, 1)].set_facecolor('#4CAF50')
            table[(i, 0)].set_text_props(weight='bold', color='white')
            table[(i, 1)].set_text_props(weight='bold', color='white')
        else:
            table[(i, 0)].set_facecolor('#f0f0f0')
            table[(i, 1)].set_facecolor('#ffffff')

    plt.tight_layout()
    return save_figure(fig, "01_parent_molecule.png")

def create_similarity_heatmap(df, top_n=None, colormap='viridis'):
    """Create comprehensive similarity heatmap."""
    df_sorted = df.sort_values('Average_Similarity', ascending=False).reset_index(drop=True)

    if top_n and len(df_sorted) > top_n:
        df_sorted = df_sorted.head(top_n)
        title_suffix = f" (Top {top_n})"
    else:
        title_suffix = f" (All {len(df_sorted)} compounds)"

    # Prepare data
    metrics = ['Tanimoto_Morgan', 'Tanimoto_FCFP', 'Tanimoto_MACCS',
               'Dice_Morgan', 'Dice_FCFP', 'Dice_MACCS']
    data = df_sorted[metrics].values

    # Create figure
    fig = plt.figure(figsize=(16, max(8, len(df_sorted) * 0.3)))
    gs = gridspec.GridSpec(2, 2, width_ratios=[3, 1], height_ratios=[1, 3],
                          hspace=0.3, wspace=0.2)

    # Title
    fig.suptitle(f'Molecular Similarity Analysis{title_suffix}',
                fontsize=20, fontweight='bold', y=0.95)

    # Statistics panel
    ax_stats = fig.add_subplot(gs[0, 0])
    ax_stats.axis('off')
    stats_text = "Statistical Summary:\n" + "="*30 + "\n"
    for metric in metrics:
        mean_val = df_sorted[metric].mean()
        std_val = df_sorted[metric].std()
        stats_text += f"{metric.replace('_', ' ')}: {mean_val:.3f} ± {std_val:.3f}\n"
    stats_text += f"Average Similarity: {df_sorted['Average_Similarity'].mean():.3f} ± {df_sorted['Average_Similarity'].std():.3f}"

    ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes,
                 fontsize=11, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))

    # Main heatmap
    ax_heat = fig.add_subplot(gs[1, 0])
    cmap = COLORMAP_OPTIONS.get(colormap, plt.cm.viridis)
    im = ax_heat.imshow(data, cmap=cmap, aspect='auto', vmin=0, vmax=1)

    # Configure heatmap
    ax_heat.set_xticks(range(len(metrics)))
    ax_heat.set_xticklabels([m.replace('_', '\n') for m in metrics],
                           rotation=0, fontweight='bold')

    compound_names = [name[:25] + '...' if len(name) > 25 else name
                     for name in df_sorted['compound_name']]
    ax_heat.set_yticks(range(len(df_sorted)))
    ax_heat.set_yticklabels(compound_names, fontsize=10)

    ax_heat.set_title('Fingerprint Similarity Matrix', fontweight='bold', pad=20)

    # Add value annotations for smaller datasets
    if len(df_sorted) <= 20:
        for i in range(len(df_sorted)):
            for j in range(len(metrics)):
                text_color = 'white' if data[i, j] < 0.5 else 'black'
                ax_heat.text(j, i, f'{data[i, j]:.2f}', ha='center', va='center',
                           color=text_color, fontweight='bold', fontsize=9)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.02)
    cbar.set_label('Similarity Score', rotation=270, labelpad=20, fontweight='bold')

    # Average similarity bar plot
    ax_bar = fig.add_subplot(gs[1, 1])
    y_pos = np.arange(len(df_sorted))
    avg_sim = df_sorted['Average_Similarity'].values

    bars = ax_bar.barh(y_pos, avg_sim, color=plt.cm.viridis(avg_sim),
                      height=0.8, edgecolor='black', linewidth=0.5)
    ax_bar.set_yticks([])
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel('Average Similarity', fontweight='bold')
    ax_bar.set_title('Compound Ranking', fontweight='bold', pad=20)
    ax_bar.set_xlim(0, 1.05)
    ax_bar.grid(axis='x', alpha=0.3)

    # Add threshold lines
    for thresh, label, color in [(0.8, 'High', 'green'), (0.6, 'Medium', 'orange'), (0.4, 'Low', 'red')]:
        ax_bar.axvline(thresh, color=color, linestyle='--', alpha=0.7, linewidth=2)
        ax_bar.text(thresh, -0.5, label, ha='center', color=color, fontweight='bold')

    # Add value labels for smaller datasets
    if len(df_sorted) <= 30:
        for i, v in enumerate(avg_sim):
            ax_bar.text(v + 0.02, i, f'{v:.2f}', va='center', fontweight='bold')

    plt.tight_layout()

    filename = f"02_similarity_heatmap{'_top' + str(top_n) if top_n else '_all'}.png"
    return save_figure(fig, filename)

def create_distribution_plots(df):
    """Create distribution analysis plots."""
    metrics = ['Tanimoto_Morgan', 'Tanimoto_FCFP', 'Tanimoto_MACCS',
               'Dice_Morgan', 'Dice_FCFP', 'Dice_MACCS', 'Average_Similarity']

    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle('Similarity Score Distributions', fontsize=20, fontweight='bold')
    axes = axes.flatten()

    colors = plt.cm.Set3(np.linspace(0, 1, len(metrics)))

    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax = axes[i]

        # Create histogram with KDE
        ax.hist(df[metric], bins=20, alpha=0.7, color=color, edgecolor='black', density=True)

        # Add KDE
        from scipy import stats
        x = np.linspace(df[metric].min(), df[metric].max(), 100)
        kde = stats.gaussian_kde(df[metric])
        ax.plot(x, kde(x), color='red', linewidth=2, label='KDE')

        # Add statistics
        mean_val = df[metric].mean()
        median_val = df[metric].median()
        std_val = df[metric].std()

        ax.axvline(mean_val, color='blue', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')
        ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.3f}')

        ax.set_title(f'{metric.replace("_", " ")}', fontweight='bold')
        ax.set_xlabel('Similarity Score', fontweight='bold')
        ax.set_ylabel('Density', fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 1)

        # Add statistics box
        stats_text = f'n = {len(df)}\nMean = {mean_val:.3f}\nStd = {std_val:.3f}\nMin = {df[metric].min():.3f}\nMax = {df[metric].max():.3f}'
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)

    # Remove empty subplot
    if len(metrics) < len(axes):
        axes[-1].remove()

    plt.tight_layout()
    return save_figure(fig, "03_distribution_analysis.png")

def create_correlation_matrix(df):
    """Create correlation matrix between different similarity metrics."""
    metrics = ['Tanimoto_Morgan', 'Tanimoto_FCFP', 'Tanimoto_MACCS',
               'Dice_Morgan', 'Dice_FCFP', 'Dice_MACCS']

    corr_matrix = df[metrics].corr()

    fig, ax = plt.subplots(figsize=(10, 8))

    # Create heatmap
    im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)

    # Set ticks and labels
    ax.set_xticks(range(len(metrics)))
    ax.set_yticks(range(len(metrics)))
    ax.set_xticklabels([m.replace('_', '\n') for m in metrics], rotation=45)
    ax.set_yticklabels([m.replace('_', '\n') for m in metrics])

    # Add correlation values
    for i in range(len(metrics)):
        for j in range(len(metrics)):
            text_color = 'white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black'
            ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center',
                   color=text_color, fontweight='bold')

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Correlation Coefficient', rotation=270, labelpad=20, fontweight='bold')

    ax.set_title('Correlation Matrix: Similarity Metrics', fontweight='bold', pad=20)
    plt.tight_layout()

    return save_figure(fig, "04_correlation_matrix.png")

def create_summary_report(df, parent_smiles):
    """Create a summary report as text file."""
    report_path = os.path.join(output_dir, "05_analysis_summary.txt")

    try:
        with open(report_path, 'w') as f:
            f.write("MOLECULAR SIMILARITY ANALYSIS REPORT\n")
            f.write("=" * 50 + "\n\n")
            f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Parent Molecule SMILES: {parent_smiles}\n")
            f.write(f"Total Compounds Analyzed: {len(df)}\n\n")

            f.write("SIMILARITY STATISTICS:\n")
            f.write("-" * 30 + "\n")

            metrics = ['Tanimoto_Morgan', 'Tanimoto_FCFP', 'Tanimoto_MACCS',
                      'Dice_Morgan', 'Dice_FCFP', 'Dice_MACCS', 'Average_Similarity']

            for metric in metrics:
                f.write(f"\n{metric.replace('_', ' ')}:\n")
                f.write(f"  Mean: {df[metric].mean():.4f}\n")
                f.write(f"  Std:  {df[metric].std():.4f}\n")
                f.write(f"  Min:  {df[metric].min():.4f}\n")
                f.write(f"  Max:  {df[metric].max():.4f}\n")

            f.write("\n\nTOP 10 MOST SIMILAR COMPOUNDS:\n")
            f.write("-" * 40 + "\n")
            top_10 = df.nlargest(10, 'Average_Similarity')
            for i, (_, row) in enumerate(top_10.iterrows(), 1):
                f.write(f"{i:2d}. {row['compound_name'][:40]:40s} | Avg Sim: {row['Average_Similarity']:.4f}\n")

            f.write("\n\nCOMPOUNDS WITH HIGH SIMILARITY (>0.7):\n")
            f.write("-" * 45 + "\n")
            high_sim = df[df['Average_Similarity'] > 0.7]
            f.write(f"Count: {len(high_sim)}\n")
            for _, row in high_sim.iterrows():
                f.write(f"- {row['compound_name']}: {row['Average_Similarity']:.4f}\n")

        print("✅ Summary report created successfully")
        return True
    except Exception as e:
        print(f"❌ Error creating summary report: {str(e)}")
        return False

print("✅ All visualization functions defined")

✅ All visualization functions defined


In [ ]:
# =============================================================================
# STEP 6: Main Analysis Pipeline
# =============================================================================

def run_enhanced_similarity_analysis():
    """Main pipeline with improved error handling and file management."""

    print("\n" + "="*60)
    print(" ENHANCED MOLECULAR SIMILARITY ANALYSIS ".center(60))
    print("="*60 + "\n")

    # Get parent molecule SMILES
    parent_smiles = input("Enter the SMILES string for your parent molecule: ").strip()

    if not parent_smiles:
        print("❌ No SMILES provided. Exiting.")
        return None

    parent_mol = Chem.MolFromSmiles(parent_smiles)
    if parent_mol is None:
        print("❌ Invalid SMILES string. Please check and try again.")
        return None

    print(f"✅ Parent molecule validated: {parent_smiles}")

    # Upload CSV file
    print("\n📁 Please upload your CSV file with 'compound_name' and 'smiles' columns:")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded. Exiting.")
        return None

    file_name = list(uploaded.keys())[0]
    print(f"📄 Processing file: {file_name}")

    # Read CSV
    try:
        df = pd.read_csv(io.BytesIO(uploaded[file_name]))
        print(f"✅ Successfully loaded {len(df)} rows")
    except Exception as e:
        print(f"❌ Error reading CSV: {str(e)}")
        return None

    # Validate columns
    required_cols = ['compound_name', 'smiles']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}")
        print(f"Available columns: {list(df.columns)}")
        return None

    print("✅ CSV validation passed")

    # Calculate similarities
    print(f"\n🧮 Calculating similarity metrics for {len(df)} compounds...")

    results = []
    invalid_count = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing compounds"):
        mol = Chem.MolFromSmiles(row['smiles'])
        if mol is None:
            invalid_count += 1
            continue

        similarities = calculate_similarity(parent_mol, mol)
        similarities['compound_name'] = row['compound_name']
        similarities['smiles'] = row['smiles']  # ADD THIS LINE TO CAPTURE SMILES
        results.append(similarities)

    if invalid_count > 0:
        print(f"⚠️  Skipped {invalid_count} compounds with invalid SMILES")

    if not results:
        print("❌ No valid compounds found. Exiting.")
        return None

    # Create results DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Average_Similarity', ascending=False).reset_index(drop=True)

    print(f"✅ Similarity calculations completed for {len(results_df)} compounds")

    # Save results CSV
    csv_path = os.path.join(output_dir, "similarity_results.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"💾 Results saved to: {csv_path}")

    # Generate visualizations
    print("\n🎨 Generating visualizations...")

    viz_success = []

    # 1. Parent molecule visualization
    viz_success.append(visualize_parent_molecule(parent_smiles))

    # 2. Full dataset heatmap
    #viz_success.append(create_similarity_heatmap(results_df))

    # 3. Top 20 heatmap (if applicable)
    if len(results_df) >= 20:
        viz_success.append(create_similarity_heatmap(results_df, top_n=20))

    # 4. Distribution analysis
    viz_success.append(create_distribution_plots(results_df))

    # 5. Correlation matrix
    viz_success.append(create_correlation_matrix(results_df))

    # 6. Summary report
    viz_success.append(create_summary_report(results_df, parent_smiles))

    successful_viz = sum(viz_success)
    print(f"✅ Generated {successful_viz} visualizations/reports successfully")

    return results_df

In [ ]:
# =============================================================================
# STEP 7: Execute Analysis and Create Download Package
# =============================================================================

def create_download_package():
    """Create and download the complete analysis package."""

    # Run the analysis
    results_df = run_enhanced_similarity_analysis()

    if results_df is None:
        print("❌ Analysis failed. No package created.")
        return

    print(f"\n📊 Analysis Summary:")
    print(f"   • Total compounds analyzed: {len(results_df)}")
    print(f"   • Highest similarity: {results_df['Average_Similarity'].max():.4f}")
    print(f"   • Average similarity: {results_df['Average_Similarity'].mean():.4f}")
    print(f"   • Compounds with >70% similarity: {len(results_df[results_df['Average_Similarity'] > 0.7])}")

    # Create ZIP package
    print(f"\n📦 Creating download package...")

    zip_filename = f"molecular_similarity_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    try:
        shutil.make_archive(zip_filename, 'zip', output_dir)
        print(f"✅ Package created: {zip_filename}.zip")

        # Download the package
        print(f"⬇️  Initiating download...")
        files.download(f"{zip_filename}.zip")

        print(f"\n🎉 Analysis completed successfully!")
        print(f"Your download package contains:")
        print(f"   • Similarity results CSV")
        print(f"   • Parent molecule visualization")
        print(f"   • Similarity heatmaps")
        print(f"   • Distribution analysis plots")
        print(f"   • Correlation matrix")
        print(f"   • Detailed summary report")

    except Exception as e:
        print(f"❌ Error creating package: {str(e)}")
        print("Files are available in the output directory for manual download.")


In [ ]:
# =============================================================================
# STEP 8: Run the Complete Analysis
# =============================================================================

print("🚀 Starting Enhanced Molecular Similarity Analysis...")
print("📋 Instructions:")
print("   1. Enter your parent molecule SMILES when prompted")
print("   2. Upload your CSV file with 'compound_name' and 'smiles' columns")
print("   3. Wait for analysis to complete")
print("   4. Download the generated ZIP package")
print("\n" + "="*60)

# Execute the complete pipeline
create_download_package()

🚀 Starting Enhanced Molecular Similarity Analysis...
📋 Instructions:
   1. Enter your parent molecule SMILES when prompted
   2. Upload your CSV file with 'compound_name' and 'smiles' columns
   3. Wait for analysis to complete
   4. Download the generated ZIP package


           ENHANCED MOLECULAR SIMILARITY ANALYSIS           

Enter the SMILES string for your parent molecule: CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O  
✅ Parent molecule validated: CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O

📁 Please upload your CSV file with 'compound_name' and 'smiles' columns:


Saving final_filtered_compounds.csv to final_filtered_compounds.csv
📄 Processing file: final_filtered_compounds.csv
✅ Successfully loaded 4172 rows
✅ CSV validation passed

🧮 Calculating similarity metrics for 4172 compounds...


Processing compounds:   0%|          | 0/4172 [00:00<?, ?it/s]

✅ Similarity calculations completed for 4172 compounds
💾 Results saved to: similarity_analysis_results/similarity_results.csv

🎨 Generating visualizations...
✅ Saved: 01_parent_molecule.png
✅ Saved: 02_similarity_heatmap_top20.png
✅ Saved: 03_distribution_analysis.png
✅ Saved: 04_correlation_matrix.png
✅ Summary report created successfully
✅ Generated 5 visualizations/reports successfully

📊 Analysis Summary:
   • Total compounds analyzed: 4172
   • Highest similarity: 0.4735
   • Average similarity: 0.2172
   • Compounds with >70% similarity: 0

📦 Creating download package...
✅ Package created: molecular_similarity_analysis_20250605_170855.zip
⬇️  Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Analysis completed successfully!
Your download package contains:
   • Similarity results CSV
   • Parent molecule visualization
   • Similarity heatmaps
   • Distribution analysis plots
   • Correlation matrix
   • Detailed summary report
